# QICK RFSoC client (JupyterLab, runs on your PC)

Talks to the ZCU216 over Pyro. The board must already be running `rfsoc_pyro_server.ipynb`.

**Kernel:** select the conda env that has `qick==0.2.422`, `Pyro4`, `psutil`. If it is missing from JupyterLab's kernel list, run once inside the env:
`pip install ipykernel` then `python -m ipykernel install --user --name qick-client --display-name "qick-client"`.

Run the cells in order. Cells 1 to 6 only define things; the "Run" sections at the bottom do the work.
**Interrupt (■ button)** stops a run that is waiting for a trigger; the `finally` blocks disarm the tProc and reset the generators.

## 1. Settings

In [15]:
import qick
from qick.asm_v2 import AveragerProgramV2
print(qick.__file__)                                       # should be under C:\qick_board
print('axis_sg_mixmux16_v1' in AveragerProgramV2.gentypes) # should be True

C:\qick_board\qick\__init__.py
True


In [17]:
import time
from dataclasses import dataclass
from typing import Callable, List, Optional

from qick.asm_v2 import AveragerProgramV2
from qick.pyro import make_proxy

NS_HOST = "192.168.2.100"     # <-- IP address of the ZCU216
# run the following command on the host to extract ip address of zcu216
#!hostname -I
#!ip -4 addr show
NS_PORT = 8888
PROXY_NAME = "myqick"

GEN_CH = 7
MIXER_FREQ = 0
TONE_FREQS = [199.2, 199.3, 199.4, 199.5, 199.6, 199.7, 199.8, 199.9,
              200.0, 200.1, 200.2, 200.3, 200.4, 200.5, 200.6, 200.7]   # MHz, absolute (tProc v2)
TONE_GAINS = [0.0625] * 16
TONE_PHASES = [0] * 16
ALL_TONES = list(range(16))

## 2. Your original notebook program

In [18]:
class MixMuxProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch, gen_ch = cfg['ro_ch'], cfg['gen_ch']
        self.declare_gen(ch=gen_ch, nqz=1, ro_ch=ro_ch, mixer_freq=cfg['mixer_freq'],
                         mux_freqs=cfg['pulse_freqs'], mux_gains=cfg['pulse_gains'],
                         mux_phases=cfg['pulse_phases'])
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['pulse_freqs'][0],
                               phase=cfg['ro_phase'], gen_ch=gen_ch)
        self.add_pulse(ch=gen_ch, name="mymixmux", style="const",
                       length=cfg["pulse_len"], mask=cfg['pulse_mask'])

    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], ddr4=True)
        self.pulse(ch=cfg['gen_ch'], name="mymixmux", t=0)


def run_original_test(soc, soccfg, pulse_len_us=4_000_000.0):
    config = {'gen_ch': GEN_CH, 'ro_ch': 0, 'mixer_freq': MIXER_FREQ,
              'pulse_freqs': TONE_FREQS, 'pulse_gains': TONE_GAINS,
              'pulse_phases': TONE_PHASES, 'pulse_mask': ALL_TONES,
              'ro_phase': 0.0, 'trig_time': 0.4, 'pulse_len': pulse_len_us, 'ro_len': 1.9}
    prog = MixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
    try:
        return prog.acquire_decimated(soc, rounds=1)
    finally:
        soc.reset_gens()

## 3. Shot builder
A shot is a list of `on(duration_us, tones=None)` and `off(duration_us)` segments. Build it with ordinary Python loops.

In [19]:
@dataclass
class Seg:
    is_on: bool
    dur_us: float
    tones: Optional[List[int]] = None     # None -> all tones; list -> only these mux tones

def on(dur_us, tones=None):
    return Seg(True, float(dur_us), tones)

def off(dur_us):
    return Seg(False, float(dur_us))

def shot_length_us(shot):
    return sum(s.dur_us for s in shot)

## 4. Program that plays one shot per iteration

In [20]:
class SequenceProgram(AveragerProgramV2):
    """cfg keys: gen_ch, mixer_freq, pulse_freqs/gains/phases, segments (list of Seg),
    gap_us (idle time after every shot), marker_pin (output pin index or None)."""

    def _initialize(self, cfg):
        g = cfg['gen_ch']
        self.declare_gen(ch=g, nqz=1, mixer_freq=cfg['mixer_freq'],
                         mux_freqs=cfg['pulse_freqs'], mux_gains=cfg['pulse_gains'],
                         mux_phases=cfg['pulse_phases'])
        for i, s in enumerate(cfg['segments']):
            if s.is_on:
                mask = s.tones if s.tones is not None else list(range(len(cfg['pulse_freqs'])))
                self.add_pulse(ch=g, name=f"seg{i}", style="const", length=s.dur_us, mask=mask)

    def _body(self, cfg):
        total = shot_length_us(cfg['segments'])
        if cfg.get('marker_pin') is not None:
            # digital marker high for the whole shot; pin index from print(soccfg)
            self.trigger(pins=[cfg['marker_pin']], t=0, width=total)
        t = 0.0
        for i, s in enumerate(cfg['segments']):
            if s.is_on:
                self.pulse(ch=cfg['gen_ch'], name=f"seg{i}", t=t)
            t += s.dur_us
        # advance the timeline past the whole shot (including trailing "off" time) plus the gap
        self.delay(total + cfg['gap_us'])


def make_sequence_program(soccfg, shot, reps=1, gap_us=1.0, marker_pin=None):
    cfg = {'gen_ch': GEN_CH, 'mixer_freq': MIXER_FREQ,
           'pulse_freqs': TONE_FREQS, 'pulse_gains': TONE_GAINS, 'pulse_phases': TONE_PHASES,
           'segments': shot, 'gap_us': gap_us, 'marker_pin': marker_pin}
    # final_delay=None: the body already advances the timeline itself
    return SequenceProgram(soccfg, reps=reps, final_delay=None, cfg=cfg)

## 5. Ways to repeat the shot

In [21]:
def run_hardware_reps(soc, soccfg, shot, n_shots, **kw):
    """Fastest: the tProc repeats the shot n_shots times back-to-back after one internal start."""
    prog = make_sequence_program(soccfg, shot, reps=n_shots, **kw)
    try:
        prog.run_rounds(soc, rounds=1, start_src="internal", progress=False)
    finally:
        soc.reset_gens()


def run_python_loop(soc, soccfg, shot, n_shots, per_shot=None, **kw):
    """One shot per Python iteration, internal start. Use when the host must act between shots."""
    prog = make_sequence_program(soccfg, shot, reps=1, **kw)
    try:
        for i in range(n_shots):
            if per_shot is not None:
                per_shot(i)
            prog.run_rounds(soc, rounds=1, start_src="internal", progress=False)
    finally:
        soc.reset_gens()


def run_external_triggered(soc, soccfg, shot, n_triggers, shots_per_trigger=1,
                           on_armed=None, on_done=None, **kw):
    """Each external trigger on the tProc start pin plays the shot (shots_per_trigger times).

    Per trigger i:  prepare_round() -> ARMED -> on_armed(i) ... 3.3 V rising edge ...
                    finish_round() -> shot played -> on_done(i)
    Triggers arriving between on_done(i) and on_armed(i+1) are MISSED (re-arm gap, printed below).
    """
    prog = make_sequence_program(soccfg, shot, reps=shots_per_trigger, **kw)
    try:
        prog.run_rounds(soc, rounds=n_triggers, start_src="external", progress=False, step_rounds=True)
        i = 0
        t_done = None
        while True:
            if on_armed is not None:
                on_armed(i)
            if t_done is not None:
                print(f"  re-arm gap after trigger {i-1}: {(time.time()-t_done)*1e3:.1f} ms")
            more = prog.finish_round()          # blocks until trigger arrives and the shot completes
            t_done = time.time()
            if on_done is not None:
                on_done(i)
            if not more:
                break
            prog.prepare_round()                # re-arm for the next trigger
            i += 1
        prog.finish_acquire()
    finally:
        soc.start_src("internal")               # disarm if interrupted while waiting
        soc.reset_gens()

---
# Run
## 6. Connect
`print(soccfg)` lists the digital output pins and the **external start pin** (where the 3.3 V trigger goes).

In [22]:
soc, soccfg = make_proxy(ns_host=NS_HOST, ns_port=NS_PORT, proxy_name=PROXY_NAME)
print(soccfg)

Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
myqick PYRO:obj_ba21cb65c8d747e4bb7c0b1a3145db6d@192.168.2.100:42331
QICK running on ZCU216, software version 0.2.422

Firmware configuration (built Sun Oct  4 15:46:57 2026):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc core clock, tProc timing clock, DAC tile 0, DAC tile 1, DAC tile 2, DAC tile 3], [ADC tile 1, ADC tile 2]

	16 signal generator channels:
	0:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		32-bit DDS, range=430.080 MHz
		DAC tile 0, blk 0 is 0_228 on JHC1, or QICK box DAC port 0
	1:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		32-bit DDS, range=430.080 MHz
		DAC tile 0, blk 1 is 1_228 on JHC2, or QICK box DAC port 1
	2:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		32-bit DDS, range=430.080 MHz
		DAC tile 0, blk 2 is 2_228 on JHC1, or QICK box DAC port 2
	3:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.08

## 7. Build a shot with Python loops

In [23]:
shot = []
for k in range(3):                                      # three 5 us pulses, 2 us apart
    shot += [on(5.0*1e5), off(2.01e5)]
shot += [on(10.0, tones=[0, 1, 2, 3]), off(5.0)]        # then a 10 us pulse on 4 tones only
print(f"shot length = {shot_length_us(shot):.1f} us")

MARKER_PIN = None    # set to an output-pin index from print(soccfg) for a shot-long marker

shot length = 2103015.0 us


## 8a. Original notebook test (4 s pulse)

In [6]:
run_original_test(soc, soccfg)

100%|██████████| 1/1 [00:00<00:00, 38.58it/s]


[array([[ 3.,  0.],
        [ 1.,  0.],
        [ 3.,  0.],
        ...,
        [ 3.,  0.],
        [-1.,  0.],
        [ 1.,  0.]], shape=(584, 2))]

## 8b. Hardware repetition (1000 shots back-to-back, internal start)

In [ ]:
run_hardware_reps(soc, soccfg, shot, n_shots=1000, marker_pin=MARKER_PIN)

## 8c. Python for-loop (one shot per iteration, internal start)

In [13]:
run_python_loop(soc, soccfg, shot, n_shots=20,
                per_shot=lambda i: print("shot", i), marker_pin=MARKER_PIN)

shot 0
shot 1


KeyboardInterrupt: 

## 8d. External trigger: each 3.3 V rising edge plays the shot once
Blocks while waiting for triggers; use the ■ interrupt button to abort. Keep trigger spacing longer than the printed re-arm gap.

In [24]:
run_external_triggered(
    soc, soccfg, shot, n_triggers=10, shots_per_trigger=1, marker_pin=MARKER_PIN,
    on_armed=lambda i: print(f"[{i}] ARMED, waiting for 3.3 V rising edge on external start pin..."),
    on_done=lambda i: print(f"[{i}] trigger received, shot played"))

[0] ARMED, waiting for 3.3 V rising edge on external start pin...


Traceback (most recent call last):
  File "C:\Users\abhim\anaconda3\envs\qick_env\Lib\site-packages\Pyro4\socketutil.py", line 171, in receiveData
    chunk = sock.recv(min(60000, size - msglen))
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ConnectionAbortedError: [WinError 10053] An established connection was aborted by the software in your host machine

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\abhim\AppData\Local\Temp\ipykernel_49104\3819190685.py", line 40, in run_external_triggered
    more = prog.finish_round()          # blocks until trigger arrives and the shot completes
           ^^^^^^^^^^^^^^^^^^^
  File "C:\qick_board\qick\qick_asm.py", line 2321, in finish_round
    new_count = soc.get_tproc_counter(addr=self.counter_addr)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\abhim\anaconda3\envs\qick_env\Lib\site-packages\Pyro4\core.py", line 185, in __call__
    r